#### Architecture:

* a) Transfer Learning

* b) Fine-tuning

* ✅ Fixed: Class Weighting + Early Stopping + Correct label order

In [ ]:
import os
import tensorflow as tf

# 1. Credentials
os.environ['KAGGLE_USERNAME'] = 'Omar/Essam/20'
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_133a7b3987b99937566e00032884d07d'

# 2. Define path
base_path = '/content/emotion-detection-fer'
os.makedirs(base_path, exist_ok=True)

# 3. Download dataset
!kaggle datasets download -d ananthu017/emotion-detection-fer --force

# 4. Unzip
!unzip -nq emotion-detection-fer.zip -d {base_path}

In [ ]:
import tensorflow as tf

data_path = '/content/emotion-detection-fer/train'

# image_dataset_from_directory assigns labels ALPHABETICALLY by folder name:
# 0=angry, 1=disgust, 2=fear, 3=happy, 4=neutral, 5=sad, 6=surprise
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    labels='inferred',
    label_mode='int',
    validation_split=0.1,
    subset='training',
    image_size=(224, 224),
    seed=42,
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.1,
    subset='validation',
    image_size=(224, 224),
    seed=42,
    batch_size=32
)

print('Class names (alphabetical order):', train_ds.class_names)
print('Confirmed order:', CLASS_NAMES)

In [ ]:
import numpy as np

# Count images per class in the train folder (alphabetical order)
# angry, disgust, fear, happy, neutral, sad, surprise
class_counts = np.array([
    len(os.listdir(os.path.join(data_path, c)))
    for c in sorted(os.listdir(data_path))
])

print('Class counts:', dict(zip(CLASS_NAMES, class_counts)))

total = class_counts.sum()
n_classes = len(class_counts)

# Inverse-frequency weighting: rare classes get higher weight
class_weights = {
    i: total / (n_classes * count)
    for i, count in enumerate(class_counts)
}

print('\nClass weights:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {class_weights[i]:.3f}')

#### a) Transfer Learning — Frozen Base

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input

base_model = ResNet50(include_top=False, weights='imagenet')
base_model.summary()

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras import Model

inputs = Input(shape=(224, 224, 3))

# Augmentation (only active during training)
img_ = RandomFlip('horizontal')(inputs)
img_ = RandomRotation(0.1)(img_)
img_ = RandomZoom(0.1)(img_)                 # ✅ added: helps generalization

# ResNet preprocessing
img_ = preprocess_input(img_)
hidden = base_model(img_)

# Head
hidden = GlobalAveragePooling2D()(hidden)
hidden = Dropout(0.5)(hidden)                # ✅ increased from 0.4 → 0.5

output = Dense(7, activation='softmax', kernel_initializer='glorot_normal')(hidden)
emotion_detection_model = Model(inputs=[inputs], outputs=[output])
emotion_detection_model.summary()

In [ ]:
# Freeze entire base model for transfer learning phase
for layer in base_model.layers:
    layer.trainable = False

emotion_detection_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

def get_callbacks(phase):
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            'emotion_model.keras',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        )
    ]

def plot_loss_acc(history, model_name='model'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].set_title(f'Loss — {model_name}')
    axes[0].plot(history.history['loss'], c='red', label='train')
    axes[0].plot(history.history['val_loss'], c='green', label='val')
    axes[0].legend()
    axes[0].set_xlabel('Epoch')

    axes[1].set_title(f'Accuracy — {model_name}')
    axes[1].plot(history.history['accuracy'], c='red', label='train')
    axes[1].plot(history.history['val_accuracy'], c='green', label='val')
    axes[1].legend()
    axes[1].set_xlabel('Epoch')

    plt.tight_layout()
    plt.show()

    best_val = max(history.history['val_accuracy'])
    print(f'Best val_accuracy: {best_val:.4f} ({best_val*100:.1f}%)')

In [ ]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.0001)
emotion_detection_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('🚀 Phase 1: Transfer Learning (frozen base)...')
history_tl = emotion_detection_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,      # ✅ fix class imbalance
    callbacks=get_callbacks('phase1')
)

plot_loss_acc(history_tl, model_name='Transfer Learning')

#### b) Fine-tuning — Unfreeze last 20 layers of ResNet

In [ ]:
# Unfreeze last 20 layers of base model
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Use a much lower LR for fine-tuning to avoid destroying pretrained weights
optimizer_ft = Adam(learning_rate=1e-5)   # ✅ lowered from 0.001 → 0.00001
emotion_detection_model.compile(
    optimizer=optimizer_ft,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

trainable_count = sum(1 for l in emotion_detection_model.layers if l.trainable)
print(f'Trainable layers: {trainable_count}')

print('\n🚀 Phase 2: Fine-tuning (last 20 layers unfrozen)...')
history_ft = emotion_detection_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights,      # ✅ keep class weights
    callbacks=get_callbacks('phase2')
)

plot_loss_acc(history_ft, model_name='Fine-tuning')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

print('📊 Evaluating per-class accuracy...')

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = emotion_detection_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print('\nClassification Report:')
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES
))

In [ ]:
# Save the best model (ModelCheckpoint already saved the best during training,
# but we also save the final state here just in case)
emotion_detection_model.save('emotion_model.keras')
print('✅ Model saved as emotion_model.keras')
print()
print('Label order for app.py:')
print('CLASS_NAMES =', CLASS_NAMES)
print()
print('Download the file and push it to your GitHub repo.')

In [ ]:
# Download the model from Colab
from google.colab import files
files.download('emotion_model.keras')